The objective here is to test the `jac` attribute of the `minimize` function of the `scipy.optimize` module.
We will consider the following optimization problem:
$$
\argmin_{(x,y,z) \in \mathbb{R}^3}\quad (x-1)^2 + y^2 + z^2
$$

In [34]:
from scipy.optimize import minimize
from time import time
from sklearn.metrics import root_mean_squared_error

Let's first define our objective function:

In [35]:
def f(theta):
    x, y, z = theta
    return (x - 1)**2 + y**2 + z**2

We will also extract the execution time to compare it with the other method (with the jacobian expression):

In [36]:
theta0 = [10, 10, 10]

In [37]:
t0 = time()
result0 = minimize(f, theta0)
exec_time0 = time() - t0
print(f"execution time = {exec_time0} s")

execution time = 0.003049135208129883 s


Let's check out the results:

In [38]:
print(result0)

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 1.3989456275134167e-11
        x: [ 1.000e+00  1.412e-06  1.412e-06]
      nit: 3
      jac: [-6.310e-06  2.839e-06  2.839e-06]
 hess_inv: [[ 8.559e-01 -1.601e-01 -1.601e-01]
            [-1.601e-01  8.221e-01 -1.779e-01]
            [-1.601e-01 -1.779e-01  8.221e-01]]
     nfev: 20
     njev: 5


Now, we know the expression of the jacobian is as follows:
$$
J(x, y, z) = \begin{pmatrix} 2 (x - 1) \\ 2y \\ 2z\end{pmatrix}
$$
We will plug it in the `jac` attribute of the `minimize` function.

In [39]:
def J(theta):
    x, y, z = theta
    return [2*(x - 1), 2*y, 2*z]

In [40]:
t0 = time()
result1 = minimize(f, theta0, jac=J)
exec_time1 = time() - t0
print(f"execution time = {exec_time1} s")

execution time = 0.0010542869567871094 s


Let's check out the **difference in execution time**:

In [41]:
print(f"execution time gain = {100*((exec_time0 - exec_time1)/exec_time0)} %")

execution time gain = 65.42341074360779 %


Now, for the results:

In [42]:
print(result1)

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 1.1044052673094165e-29
        x: [ 1.000e+00 -2.665e-15 -1.776e-15]
      nit: 2
      jac: [ 1.776e-15 -5.329e-15 -3.553e-15]
 hess_inv: [[ 8.559e-01 -1.601e-01 -1.601e-01]
            [-1.601e-01  8.221e-01 -1.779e-01]
            [-1.601e-01 -1.779e-01  8.221e-01]]
     nfev: 4
     njev: 4


We will checkout the rmse between the true and estimated values for the two methods:

In [43]:
theta_true = [1, 0, 0]
print(f"rmse with auto jacobian = {root_mean_squared_error(theta_true, result0.x)}")
print(f"rmse with defined jacobian = {root_mean_squared_error(theta_true, result1.x)}")

rmse with auto jacobian = 2.1594332802176104e-06
rmse with defined jacobian = 1.9186846773327266e-15


**Conclusion**: when we include a predefined jacobian, the result and execution time are better.